# Modern Data Lakehouse: Interactive Seeding & Data Federation

This notebook provides an end-to-end Python interface to:
1. **Provision Unity Catalog & Schemas** via Unity Catalog's REST API.
2. **Ingest Apache Iceberg tables** (`clickstream` and `bonuses`) into **MinIO S3** using `PyIceberg` and `PyArrow`.
3. **Register Iceberg metadata** in Unity Catalog's metastore.
4. **Execute federated cross-catalog SQL queries** using **Trino** across PostgreSQL and Unity Catalog/MinIO.
5. **Visualize analytical insights** directly in Python using Pandas and Matplotlib.

### Step 0: Ensure Kubernetes Port-Forwarding is Active
Make sure the following services are forwarded from your Kubernetes cluster in separate terminals:
```bash
kubectl port-forward svc/trino 8080:8080 -n lakehouse
kubectl port-forward svc/minio 9000:9000 -n lakehouse
kubectl port-forward svc/unitycatalog 8083:8080 -n lakehouse
```

In [ ]:
# Cell 1: Import core dependencies
import os
import sys
import uuid
import subprocess
from datetime import datetime, timezone

import requests
import pandas as pd
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, TimestamptzType, DoubleType, IntegerType
from minio import Minio
from trino.dbapi import connect
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

--- 
## 1. Create Catalog and Schemas via Unity Catalog REST API
We use the REST API of Unity Catalog (`http://localhost:8083`) to register the catalog and schemas.

In [ ]:
# Cell 2: Create Catalog & Schemas in Unity Catalog
UC_BASE = "http://localhost:8083/api/2.1/unity-catalog"

# 1. Create 'unity' Catalog
res = requests.post(f"{UC_BASE}/catalogs", json={"name": "unity", "comment": "Main Lakehouse Catalog"})
print("Catalog 'unity':", res.status_code, res.json() if res.status_code == 200 else res.text)

# 2. Create 'analytics_schema'
res = requests.post(f"{UC_BASE}/schemas", json={"name": "analytics_schema", "catalog_name": "unity"})
print("Schema 'analytics_schema':", res.status_code, res.json() if res.status_code == 200 else res.text)

# 3. Create 'finance_schema'
res = requests.post(f"{UC_BASE}/schemas", json={"name": "finance_schema", "catalog_name": "unity"})
print("Schema 'finance_schema':", res.status_code, res.json() if res.status_code == 200 else res.text)

--- 
## 2. Ingest Apache Iceberg Tables into MinIO S3
We use `PyIceberg` to create native Iceberg tables on MinIO (`s3://warehouse/`), write Parquet data files, and generate metadata manifests.

In [ ]:
# Cell 3: Connect PyIceberg to MinIO and ingest clickstream & bonuses data
catalog = SqlCatalog(
    "lakehouse",
    **{
        "uri": "sqlite:///:memory:",
        "warehouse": "s3://warehouse",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
        "s3.region": "us-east-1",
    },
)

minio_client = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)

# --- Table 1: clickstream ---
try:
    catalog.create_namespace("analytics_schema")
except Exception:
    pass

click_schema = Schema(
    NestedField(field_id=1, name="event_id", field_type=StringType(), required=True),
    NestedField(field_id=2, name="user_id", field_type=StringType(), required=True),
    NestedField(field_id=3, name="event_type", field_type=StringType(), required=True),
    NestedField(field_id=4, name="page_url", field_type=StringType(), required=True),
    NestedField(field_id=5, name="event_timestamp", field_type=TimestamptzType(), required=True),
)

try:
    catalog.drop_table(("analytics_schema", "clickstream"))
except Exception:
    pass

click_table = catalog.create_table(
    identifier=("analytics_schema", "clickstream"),
    schema=click_schema,
    location="s3://warehouse/analytics_schema/clickstream",
)

click_data = [
    ("evt_001", "usr_001", "page_view", "/home", datetime(2026, 9, 12, 10, 15, 0, tzinfo=timezone.utc)),
    ("evt_002", "usr_001", "click", "/products", datetime(2026, 9, 12, 10, 18, 22, tzinfo=timezone.utc)),
    ("evt_003", "usr_001", "click", "/cart", datetime(2026, 9, 12, 10, 20, 5, tzinfo=timezone.utc)),
    ("evt_010", "usr_001", "purchase", "/thank-you", datetime(2026, 9, 12, 10, 25, 0, tzinfo=timezone.utc)),
    ("evt_004", "usr_002", "page_view", "/home", datetime(2026, 9, 12, 11, 0, 10, tzinfo=timezone.utc)),
    ("evt_005", "usr_002", "click", "/pricing", datetime(2026, 9, 12, 11, 5, 40, tzinfo=timezone.utc)),
    ("evt_006", "usr_003", "page_view", "/blog", datetime(2026, 9, 12, 12, 30, 15, tzinfo=timezone.utc)),
    ("evt_007", "usr_004", "page_view", "/login", datetime(2026, 9, 12, 13, 0, 0, tzinfo=timezone.utc)),
    ("evt_008", "usr_005", "page_view", "/products", datetime(2026, 9, 12, 14, 10, 0, tzinfo=timezone.utc)),
    ("evt_009", "usr_005", "click", "/checkout", datetime(2026, 9, 12, 14, 15, 30, tzinfo=timezone.utc)),
]

click_arrow = pa.Table.from_arrays(
    [
        pa.array([r[0] for r in click_data], type=pa.string()),
        pa.array([r[1] for r in click_data], type=pa.string()),
        pa.array([r[2] for r in click_data], type=pa.string()),
        pa.array([r[3] for r in click_data], type=pa.string()),
        pa.array([r[4] for r in click_data], type=pa.timestamp("us", tz="UTC")),
    ],
    schema=click_table.schema().as_arrow(),
)
click_table.append(click_arrow)
print("Clickstream table created in MinIO:", click_table.metadata_location)

# --- Table 2: bonuses ---
try:
    catalog.create_namespace("finance_schema")
except Exception:
    pass

bonus_schema = Schema(
    NestedField(field_id=1, name="emp_id", field_type=StringType(), required=True),
    NestedField(field_id=2, name="annual_bonus", field_type=DoubleType(), required=True),
    NestedField(field_id=3, name="performance_score", field_type=DoubleType(), required=True),
    NestedField(field_id=4, name="fiscal_year", field_type=IntegerType(), required=True),
)

try:
    catalog.drop_table(("finance_schema", "bonuses"))
except Exception:
    pass

bonus_table = catalog.create_table(
    identifier=("finance_schema", "bonuses"),
    schema=bonus_schema,
    location="s3://warehouse/finance_schema/bonuses",
)

bonus_data = [
    ("EMP001", 18500.0, 4.8, 2026),
    ("EMP002", 14200.0, 4.5, 2026),
    ("EMP003", 16000.0, 4.7, 2026),
    ("EMP004", 15500.0, 4.6, 2026),
    ("EMP005", 9800.0, 4.1, 2026),
]

bonus_arrow = pa.Table.from_arrays(
    [
        pa.array([r[0] for r in bonus_data], type=pa.string()),
        pa.array([r[1] for r in bonus_data], type=pa.float64()),
        pa.array([r[2] for r in bonus_data], type=pa.float64()),
        pa.array([r[3] for r in bonus_data], type=pa.int32()),
    ],
    schema=bonus_table.schema().as_arrow(),
)
bonus_table.append(bonus_arrow)
print("Bonuses table created in MinIO:", bonus_table.metadata_location)

--- 
## 3. Register Tables in Unity Catalog
We copy the generated Iceberg metadata manifests to Unity Catalog and register the tables in the `unity_catalog` PostgreSQL metastore.

In [ ]:
# Cell 4: Register Iceberg tables in Unity Catalog
uc_pod = subprocess.check_output(
    ["kubectl", "get", "pods", "-n", "lakehouse", "-l", "app=unitycatalog", "-o", "jsonpath={.items[0].metadata.name}"]
).decode("utf-8").strip()

# 1. Copy Clickstream metadata to UC pod
click_meta_key = click_table.metadata_location.replace("s3://warehouse/", "")
click_res = minio_client.get_object("warehouse", click_meta_key)
click_meta_bytes = click_res.read()
click_res.close()

remote_click_dir = "/home/unitycatalog/etc/data/external/unity/analytics_schema/tables/clickstream/metadata"
remote_click_file = f"{remote_click_dir}/00001.metadata.json"
subprocess.run(["kubectl", "exec", "-n", "lakehouse", uc_pod, "--", "mkdir", "-p", remote_click_dir], check=True)
subprocess.run(
    ["kubectl", "exec", "-i", "-n", "lakehouse", uc_pod, "--", "sh", "-c", f"cat > {remote_click_file}"],
    input=click_meta_bytes,
    check=True,
)

# 2. Copy Bonuses metadata to UC pod
bonus_meta_key = bonus_table.metadata_location.replace("s3://warehouse/", "")
bonus_res = minio_client.get_object("warehouse", bonus_meta_key)
bonus_meta_bytes = bonus_res.read()
bonus_res.close()

remote_bonus_dir = "/home/unitycatalog/etc/data/external/unity/finance_schema/tables/bonuses/metadata"
remote_bonus_file = f"{remote_bonus_dir}/00001.metadata.json"
subprocess.run(["kubectl", "exec", "-n", "lakehouse", uc_pod, "--", "mkdir", "-p", remote_bonus_dir], check=True)
subprocess.run(
    ["kubectl", "exec", "-i", "-n", "lakehouse", uc_pod, "--", "sh", "-c", f"cat > {remote_bonus_file}"],
    input=bonus_meta_bytes,
    check=True,
)

# 3. Register in PostgreSQL unity_catalog database
psql_reg = f"""
DO $$
DECLARE
    v_catalog_id uuid;
    v_analytics_id uuid;
    v_finance_id uuid;
BEGIN
    SELECT id INTO v_catalog_id FROM uc_catalogs WHERE name = 'unity';
    SELECT id INTO v_analytics_id FROM uc_schemas WHERE name = 'analytics_schema' AND catalog_id = v_catalog_id;
    SELECT id INTO v_finance_id FROM uc_schemas WHERE name = 'finance_schema' AND catalog_id = v_catalog_id;

    -- uc_tables clickstream
    DELETE FROM uc_tables WHERE name = 'clickstream' AND schema_id = v_analytics_id;
    INSERT INTO uc_tables (
        id, name, column_count, data_source_format, schema_id, type,
        uniform_iceberg_metadata_location, url, created_at, updated_at
    ) VALUES (
        gen_random_uuid(), 'clickstream', 0, 'DELTA', v_analytics_id, 'EXTERNAL',
        'file://{remote_click_file}', 's3://warehouse/analytics_schema/clickstream', NOW(), NOW()
    );

    -- uc_tables bonuses
    DELETE FROM uc_tables WHERE name = 'bonuses' AND schema_id = v_finance_id;
    INSERT INTO uc_tables (
        id, name, column_count, data_source_format, schema_id, type,
        uniform_iceberg_metadata_location, url, created_at, updated_at
    ) VALUES (
        gen_random_uuid(), 'bonuses', 0, 'DELTA', v_finance_id, 'EXTERNAL',
        'file://{remote_bonus_file}', 's3://warehouse/finance_schema/bonuses', NOW(), NOW()
    );
END $$;
"""

subprocess.run(
    ["kubectl", "exec", "-i", "-n", "lakehouse", "deployment/postgres", "--", "psql", "-U", "postgres", "-d", "unity_catalog"],
    input=psql_reg.encode("utf-8"),
    check=True,
)
print("Tables successfully registered in Unity Catalog!")

--- 
## 4. Federated Cross-Catalog SQL with Trino & Pandas
Now we query Trino across PostgreSQL (`postgresql.public.users`) and Unity Catalog/MinIO (`unity.analytics_schema.clickstream`) in a single query.

In [ ]:
# Cell 5: Helper function & Cross-Catalog Federated Query
def run_query(sql, conn):
    cur = conn.cursor()
    cur.execute(sql.strip().rstrip(';'))
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

conn = connect(
    host="localhost",
    port=8080,
    user="admin",
    catalog="unity",
    schema="analytics_schema",
)

query = """
SELECT 
    u.user_id,
    u.first_name,
    u.last_name,
    COUNT(c.event_id) AS total_clicks,
    MAX(c.event_timestamp) AS last_seen_date
FROM 
    postgresql.public.users u
JOIN 
    unity.analytics_schema.clickstream c ON u.user_id = c.user_id
WHERE 
    u.status = 'ACTIVE'
GROUP BY 
    u.user_id, u.first_name, u.last_name
ORDER BY 
    total_clicks DESC
"""

df_clicks = run_query(query, conn)
display(df_clicks)

--- 
## 5. Sensitive Compensation Query (Salaries + Bonuses)
Joining PostgreSQL operational salaries with object storage bonuses.

In [ ]:
# Cell 6: Total Compensation Join
payroll_query = """
SELECT 
    s.emp_id,
    s.employee_name,
    s.department,
    s.base_salary,
    b.annual_bonus,
    (s.base_salary + b.annual_bonus) AS total_compensation,
    b.performance_score
FROM 
    postgresql.public.salaries s
JOIN 
    unity.finance_schema.bonuses b ON s.emp_id = b.emp_id
ORDER BY 
    total_compensation DESC
"""

df_payroll = run_query(payroll_query, conn)
display(df_payroll)

--- 
## 6. Visualizing Analytical Insights
Plotting user engagement metrics directly from the federated dataset.

In [ ]:
# Cell 7: Plot Clickstream Activity by User
plt.figure(figsize=(9, 4.5))
bars = plt.bar(
    df_clicks["first_name"] + " " + df_clicks["last_name"],
    df_clicks["total_clicks"],
    color="#1f77b4",
    edgecolor="black",
)
plt.title("Total Web Activity Clicks per Active User (Federated Join)", fontsize=13, fontweight="bold")
plt.xlabel("Active User", fontsize=11)
plt.ylabel("Event / Click Count", fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.7)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1, f"{int(height)}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()